In [1]:
import pandas as pd
import pycountry

In [2]:
# Load population and birth-rate datasets
pop = pd.read_csv("../data/raw/pop_total.csv")
rate = pd.read_csv("../data/raw/birth_rates.csv")

In [3]:
# Remove metadata columns not required for analysis
pop = pop.drop(columns=["Series Name", "Series Code"])
rate = rate.drop(columns=["Series Name", "Series Code"])

In [4]:
# Drop rows with missing values
pop = pop.dropna()
rate = rate.dropna()

In [1]:
# Reshape datasets from wide to long format
pop_melt = pop.melt(id_vars=["Country Name", "Country Code"],
                   var_name= "Year",
                   value_name= "Population")

birth_melt = rate.melt(id_vars=["Country Name", "Country Code"],
                   var_name= "Year",
                   value_name= "BirthRate")

NameError: name 'pop' is not defined

In [ ]:
po

In [6]:
# Extract year values and convert them to integers
birth_melt['Year'] = birth_melt["Year"].str.extract(r'(\d+)').astype(int)
pop_melt['Year'] = pop_melt["Year"].str.extract(r'(\d+)').astype(int)

In [7]:
# Merge population and birth-rate data
df = pd.merge(
    pop_melt,
    birth_melt,
    on= ["Country Name", "Country Code", "Year"],
    how= "inner"
)

In [8]:
# Keep records up to 2020
df = df[df["Year"] < 2021]

In [9]:
# Filter out aggregated regions and keep valid countries only
valid_codes = {
    country.alpha_3
    for country in pycountry.countries
}
df = df[df["Country Code"].isin(valid_codes)]

In [10]:
# Convert numeric columns to appropriate data types
df["Population"] = pd.to_numeric(df["Population"], errors= "coerce")
df["BirthRate"] = pd.to_numeric(df["BirthRate"], errors= "coerce")

In [11]:
# Estimate annual births for each country
df["Births"] = (df["BirthRate"] / 1000) * df["Population"]
# Calculate total births for each year
total_birth = df.groupby("Year")["Births"].transform("sum")

In [12]:
# Calculate each country's share of global births
df["BirthProbability"] = (df["Births"] / total_birth * 100).round(3)
df["Rank"] = (
    df.groupby("Year")["BirthProbability"]
      .rank(ascending=False)
      .astype(int)
)

In [13]:
plot_df = df[
    [
        "Country Name",
        "Country Code",
        "Year",
        "Births",
        "BirthProbability",
        "Rank"
    ]
].copy()

In [14]:
plot_df.to_csv(
    "../data/processed/birth_probability.csv",
    index=False
)